In [ ]:
import audioio as AudioIO
import numpy as np
def kick(beats):
    return ("kick", beats)
def snare(beats):
    return ("snare", beats)
def ch(beats):
    return ("ch", beats)
def oh(beats):
    return ("oh", beats)

# Synthesizing a ROLAND TR-707 drum machine
## understanding the ROLAND TR-707 drum machine and thus the project
To understand this project it might be nice to understand what the TR-707 drum machine is and more importantly what it does. The TR-707 drum machine as the name might suggest is a machine that synthesizes the sound of a particular drum set making the TR-707 drum machine sound very iconic and recognizable. Each part of the drum kit weather it being kick, snare, hi-hat closed or open takes in two values they will need to know how often it is sampled and how long it should play.
## Usefulness
This project provides easier access to technology such as the ROLAND TR-707 drum machine which is a physical machine one would have to go out and buy if they can even find one today. *they have become antique*


In [ ]:
def kick(Fs, duration=0.2):
    t = np.linspace(0, duration, int(Fs * duration), False)

    # Pitch sweep (classic analog kick)
    f_start = 120
    f_end = 50
    freq = np.linspace(f_start, f_end, len(t))

    phase = 2 * np.pi * np.cumsum(freq) / Fs
    y = np.sin(phase)

    # Fast decay envelope
    envelope = np.exp(-t * 20)

    return y * envelope

def snare(Fs, duration=0.25):
    t = np.linspace(0, duration, int(Fs * duration), False)

    # Noise (the "snap")
    noise = np.random.randn(len(t))

    # Tone body
    tone = np.sin(2 * np.pi * 180 * t)

    # Envelope
    envelope = np.exp(-t * 15)

    return (0.7 * noise + 0.3 * tone) * envelope

def hihat_closed(Fs, duration=0.05):
    t = np.linspace(0, duration, int(Fs * duration), False)

    noise = np.random.randn(len(t))

    envelope = np.exp(-t * 80)

    return noise * envelope

def highpass(signal, alpha=0.9):
    y = np.zeros_like(signal)
    for i in range(1, len(signal)):
        y[i] = alpha * (y[i-1] + signal[i] - signal[i-1])
    return y

def hihat_open(Fs, duration=0.25):
    t = np.linspace(0, duration, int(Fs * duration), False)

    # Metallic partials (more inharmonic, bell-like)
    freqs = [500, 900, 1400, 2200, 2800]  # inharmonic ratios
    metallic = sum(np.sin(2 * np.pi * f * t) for f in freqs)

    # Apply exponential decay (faster for higher freqs)
    decay = np.exp(-t * 20)
    metallic *= decay

    # White noise tail
    noise = np.random.randn(len(t))
    noise_env = np.exp(-t * 12)  # slightly faster decay than metallic
    noise = noise * noise_env

    # Mix metallic + noise
    y = 0.6 * metallic + 0.4 * noise

    # Optional: high-pass for brightness (sharp attack)
    y = highpass(y, alpha=0.73)

    # Soft clipping for grit
    y = np.tanh(y * 3.0)

    # Normalize
    y /= np.max(np.abs(y) + 1e-8)

    return y

def build_beat(events, Fs, seconds_per_beat):
    total_beats = sum(beats for _, beats in events)
    total_samples = int(total_beats * seconds_per_beat * Fs)

    output = np.zeros(total_samples)
    cursor = 0

    for drum, beats in events:
        duration = beats * seconds_per_beat
        samples = int(duration * Fs)

        if drum == "kick":
            y = kick(Fs, duration)
        elif drum == "snare":
            y = snare(Fs, duration)
        elif drum == "ch":
            y = hihat_closed(Fs, duration)
        elif drum == "oh":
            y = hihat_open(Fs, duration)
        else:
            y = np.zeros(samples)

        output[cursor:cursor+samples] += y[:samples]
        cursor += samples

    return output